# 03 — Tokenização e chunking com BERTimbau

Este notebook transforma as reuniões limpas em chunks compatíveis com o BERTimbau, preservando reunião, ordem, locutores e posições no texto original.

**Objetivo e conexão com o projeto.** A entrada são as reuniões limpas; a saída são janelas ordenadas que respeitam o limite real de tokens do BERTimbau e mantêm a rastreabilidade até a reunião original. O chunk é a unidade técnica de classificação, enquanto a reunião continua sendo a unidade de negócio. Preservar essa distinção evita que limitações do modelo alterem o significado da análise.


## 1. Decisões tomadas

- Usamos `neuralmind/bert-base-portuguese-cased`, o BERTimbau Base para português brasileiro.
- O modelo admite 512 posições. Reservamos 2 para `[CLS]` e `[SEP]`, deixando 510 tokens de conteúdo.
- A sobreposição-alvo é de 64 tokens e reutiliza palavras completas.
- Os limites são definidos pelos tokens reais do tokenizer, mas cada chunk começa e termina em fronteira de palavra.
- O texto do chunk é recortado diretamente da transcrição normalizada; não usamos `decode`, evitando alterações artificiais no texto.
- Cada chunk mantém `meeting_id`, índice, offsets de caracteres, intervalo de turnos e locutores.
- Nenhuma transcrição ou trecho é exibido nas saídas do notebook.
- Os chunks permanecem em `data/processed/`, fora do Git. Futuras divisões de treino/teste serão agrupadas por reunião.

Referências: [BERTimbau Base](https://huggingface.co/neuralmind/bert-base-portuguese-cased) e [API de tokenização](https://huggingface.co/docs/transformers/main_classes/tokenizer).


## 2. Preparação do ambiente

Nesta célula são reunidos imports, parâmetros e caminhos necessários à etapa. Centralizar essa preparação antes do processamento deixa as dependências explícitas e evita que configurações importantes fiquem dispersas entre transformações, treinamento ou avaliação.


In [1]:
from __future__ import annotations

import json, os, re, tempfile
from collections import Counter
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterator

import transformers
from transformers import AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'meetings.jsonl'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_bertimbau.jsonl'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'metrics' / 'chunking_summary.json'
CHECKPOINT = 'neuralmind/bert-base-portuguese-cased'
MAX_SEQUENCE_LENGTH = 512
CONTENT_MAX_TOKENS = 510
OVERLAP_TOKENS = 64
SPEAKER_PATTERN = re.compile(r'\[LOCUTOR\s+(\d+)\]:')

C:\Users\Gabriel Pereira\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 3. Carregar e conferir o tokenizer

Baixamos apenas os arquivos do tokenizer. O modelo neural completo será necessário somente no fine-tuning.

A célula seguinte reúne somente a leitura e a conferência dos insumos desta etapa. Validar caminhos, formato e contrato antes das transformações torna falhas de entrada explícitas e evita resultados parciais difíceis de diagnosticar.


In [2]:
tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT, use_fast=True, do_lower_case=False
)
special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
assert special_tokens == 2
assert CONTENT_MAX_TOKENS + special_tokens <= MAX_SEQUENCE_LENGTH
{
    'checkpoint': CHECKPOINT,
    'transformers_version': transformers.__version__,
    'tokenizer_class': type(tokenizer).__name__,
    'is_fast': tokenizer.is_fast,
    'special_tokens': special_tokens,
    'content_max_tokens': CONTENT_MAX_TOKENS,
    'overlap_tokens_target': OVERLAP_TOKENS,
}

{'checkpoint': 'neuralmind/bert-base-portuguese-cased',
 'transformers_version': '5.17.0',
 'tokenizer_class': 'BertTokenizer',
 'is_fast': True,
 'special_tokens': 2,
 'content_max_tokens': 510,
 'overlap_tokens_target': 64}

## 4. Funções de turnos e janelas de palavras

Os offsets retornados pelo tokenizer são associados às palavras e aos turnos no texto original. A sobreposição repete apenas contexto; ela nunca cria mistura entre reuniões.


In [ ]:
def iter_ndjson(path: Path) -> Iterator[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as source:
        for line_number, line in enumerate(source, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'Registro inválido na linha {line_number}.')
            yield record

def speaker_spans(text: str) -> tuple[list[dict[str, Any]], bool]:
    matches = list(SPEAKER_PATTERN.finditer(text))
    spans = []
    has_preamble = bool(matches and text[:matches[0].start()].strip())
    if not matches:
        return ([{'turn_index': 0, 'speaker': None, 'char_start': 0, 'char_end': len(text)}], False)
    if has_preamble:
        spans.append({'turn_index': 0, 'speaker': None, 'char_start': 0, 'char_end': matches[0].start()})
    index_offset = len(spans)
    for index, match in enumerate(matches):
        spans.append({
            'turn_index': index + index_offset,
            'speaker': f'LOCUTOR {match.group(1)}',
            'char_start': match.start(),
            'char_end': matches[index + 1].start() if index + 1 < len(matches) else len(text),
        })
    return spans, has_preamble


### Funções auxiliares da seção

Esta célula agrupa `word_spans_with_token_counts`, `intersecting_turns`, definições relacionadas que permanecem dentro de um bloco legível. Elas preparam operações específicas para as células seguintes e não processam dados até serem chamadas.


In [ ]:
def word_spans_with_token_counts(text: str) -> tuple[list[tuple[int, int]], list[int], int]:
    words = [(match.start(), match.end()) for match in re.finditer(r'\S+', text)]
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = encoded['offset_mapping']
    counts = []
    token_index = 0
    for start, end in words:
        while token_index < len(offsets) and offsets[token_index][1] <= start:
            token_index += 1
        cursor = token_index
        count = 0
        while cursor < len(offsets) and offsets[cursor][0] < end:
            if offsets[cursor][1] > start:
                count += 1
            cursor += 1
        counts.append(count)
        token_index = cursor
    return words, counts, len(encoded['input_ids'])

def intersecting_turns(turns, char_start, char_end):
    return [turn for turn in turns if turn['char_end'] > char_start and turn['char_start'] < char_end]


### Construção dos chunks de uma reunião

A função cria janelas dentro do limite do BERTimbau, aplica a sobreposição-alvo e preserva ordem, offsets, turnos e locutores para cada chunk.


In [3]:
def chunk_meeting(record: dict[str, Any]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    meeting_id = str(record['ID_MEETING'])
    text = record['ANON_TRANSCRICAO']
    turns, has_preamble = speaker_spans(text)
    words, word_token_counts, meeting_tokens = word_spans_with_token_counts(text)
    turn_token_counts = [0] * len(turns)
    turn_cursor = 0
    for (word_start, _), token_count in zip(words, word_token_counts):
        while turn_cursor + 1 < len(turns) and word_start >= turns[turn_cursor]['char_end']:
            turn_cursor += 1
        turn_token_counts[turn_cursor] += token_count
    if not words:
        raise ValueError(f'Reunião {meeting_id!r} sem palavras.')
    if max(word_token_counts, default=0) > CONTENT_MAX_TOKENS:
        raise ValueError(f'Reunião {meeting_id!r} contém palavra maior que o limite do modelo.')

    chunks = []
    start_word = 0
    while start_word < len(words):
        end_word = start_word
        estimated_tokens = 0
        while end_word < len(words) and estimated_tokens + word_token_counts[end_word] <= CONTENT_MAX_TOKENS:
            estimated_tokens += word_token_counts[end_word]
            end_word += 1
        if end_word == start_word:
            end_word += 1
        char_start = words[start_word][0]
        char_end = words[end_word - 1][1]
        chunk_text = text[char_start:char_end]
        actual_tokens = len(tokenizer.encode(chunk_text, add_special_tokens=False))
        while actual_tokens > CONTENT_MAX_TOKENS and end_word > start_word + 1:
            end_word -= 1
            char_end = words[end_word - 1][1]
            chunk_text = text[char_start:char_end]
            actual_tokens = len(tokenizer.encode(chunk_text, add_special_tokens=False))
        chunk_turns = intersecting_turns(turns, char_start, char_end)
        chunk_index = len(chunks)
        chunks.append({
            'meeting_id': meeting_id,
            'chunk_id': f'{meeting_id}::chunk-{chunk_index:04d}',
            'chunk_index': chunk_index,
            'text': chunk_text,
            'num_tokens': actual_tokens,
            'char_start': char_start,
            'char_end': char_end,
            'turn_start': chunk_turns[0]['turn_index'],
            'turn_end': chunk_turns[-1]['turn_index'],
            'speakers': sorted({turn['speaker'] for turn in chunk_turns if turn['speaker']}),
            'checkpoint': CHECKPOINT,
            'content_max_tokens': CONTENT_MAX_TOKENS,
            'overlap_tokens_target': OVERLAP_TOKENS,
        })
        if end_word >= len(words):
            break
        overlap_start = end_word
        overlap_count = 0
        while overlap_start > start_word and overlap_count + word_token_counts[overlap_start - 1] <= OVERLAP_TOKENS:
            overlap_start -= 1
            overlap_count += word_token_counts[overlap_start]
        start_word = overlap_start if overlap_start > start_word else end_word

    covered_turns = {index for chunk in chunks for index in range(chunk['turn_start'], chunk['turn_end'] + 1)}
    assert len(covered_turns) == len(turns)
    assert all(0 < chunk['num_tokens'] <= CONTENT_MAX_TOKENS for chunk in chunks)
    assert [chunk['chunk_index'] for chunk in chunks] == list(range(len(chunks)))
    return chunks, {
        'meeting_tokens': meeting_tokens,
        'speaker_turns': len(turns),
        'turn_token_counts': turn_token_counts,
        'has_speaker_marker': bool(SPEAKER_PATTERN.search(text)),
        'has_preamble': has_preamble,
    }


## 5. Gerar chunks e relatório

A escrita é atômica. Se alguma reunião falhar nas validações, o arquivo anterior não é substituído por uma saída parcial.

O processamento em chunks resolve o limite de contexto sem perder a ligação com a reunião. Índices, offsets e sobreposição permitem reconstruir a ordem das evidências e agregar previsões posteriormente.


In [ ]:
def percentile(values, probability):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * probability)]

def write_json_atomic(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, prefix=f'.{path.name}.', suffix='.tmp', delete=False) as handle:
        json.dump(content, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
        temporary_path = Path(handle.name)
    os.replace(temporary_path, path)


### Leitura ou escrita dos artefatos

A operação de arquivo fica isolada nesta célula para tornar claro quais dados entram ou saem da etapa e em que momento as validações são aplicadas.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_handle = tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=OUTPUT_PATH.parent, prefix=f'.{OUTPUT_PATH.name}.', suffix='.tmp', delete=False)
temporary_path = Path(temporary_handle.name)
meeting_token_counts, turn_token_counts, chunks_per_meeting, chunk_token_counts = [], [], [], []
meetings_without_marker = meetings_with_preamble = total_meetings = total_chunks = 0
try:
    with temporary_handle as destination:
        for meeting in iter_ndjson(INPUT_PATH):
            chunks, audit = chunk_meeting(meeting)
            total_meetings += 1
            total_chunks += len(chunks)
            meeting_token_counts.append(audit['meeting_tokens'])
            turn_token_counts.extend(audit['turn_token_counts'])
            chunks_per_meeting.append(len(chunks))
            chunk_token_counts.extend(chunk['num_tokens'] for chunk in chunks)
            meetings_without_marker += not audit['has_speaker_marker']
            meetings_with_preamble += audit['has_preamble']
            metadata = {key: value for key, value in meeting.items() if key != 'ANON_TRANSCRICAO'}
            for chunk in chunks:
                destination.write(json.dumps({**metadata, **chunk}, ensure_ascii=False) + '\n')
    os.replace(temporary_path, OUTPUT_PATH)
except Exception:
    temporary_path.unlink(missing_ok=True)
    raise


### Consolidação do relatório da etapa

Configurações, contagens, métricas e alertas são reunidos em um resumo auditável. O relatório registra resultados agregados sem acrescentar novas transformações aos dados.


In [4]:
summary = {
    'checkpoint': CHECKPOINT,
    'transformers_version': transformers.__version__,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'content_max_tokens': CONTENT_MAX_TOKENS,
    'overlap_tokens_target': OVERLAP_TOKENS,
    'input_meetings': total_meetings,
    'output_chunks': total_chunks,
    'meetings_without_speaker_marker': meetings_without_marker,
    'meetings_with_text_before_first_marker': meetings_with_preamble,
    'meeting_tokens_mean': round(mean(meeting_token_counts), 2),
    'meeting_tokens_median': median(meeting_token_counts),
    'meeting_tokens_p95': percentile(meeting_token_counts, 0.95),
    'turn_tokens_mean': round(mean(turn_token_counts), 2),
    'turn_tokens_median': median(turn_token_counts),
    'turn_tokens_p95': percentile(turn_token_counts, 0.95),
    'turn_tokens_max': max(turn_token_counts),
    'chunks_per_meeting_mean': round(mean(chunks_per_meeting), 2),
    'chunks_per_meeting_p95': percentile(chunks_per_meeting, 0.95),
    'chunks_per_meeting_max': max(chunks_per_meeting),
    'chunk_tokens_mean': round(mean(chunk_token_counts), 2),
    'chunk_tokens_max': max(chunk_token_counts),
}
write_json_atomic(REPORT_PATH, summary)
summary


{'checkpoint': 'neuralmind/bert-base-portuguese-cased',
 'transformers_version': '5.17.0',
 'max_sequence_length': 512,
 'content_max_tokens': 510,
 'overlap_tokens_target': 64,
 'input_meetings': 1126,
 'output_chunks': 29972,
 'meetings_without_speaker_marker': 48,
 'meetings_with_text_before_first_marker': 100,
 'meeting_tokens_mean': 11715.28,
 'meeting_tokens_median': 10453.5,
 'meeting_tokens_p95': 25944,
 'turn_tokens_mean': 39.39,
 'turn_tokens_median': 16,
 'turn_tokens_p95': 121,
 'turn_tokens_max': 33308,
 'chunks_per_meeting_mean': 26.62,
 'chunks_per_meeting_p95': 59,
 'chunks_per_meeting_max': 130,
 'chunk_tokens_mean': 500.88,
 'chunk_tokens_max': 510}

## 6. Validação final sem conteúdo sensível

Relemos o arquivo e confirmamos IDs únicos, limites de tokens, ordem dos chunks e agrupamento contíguo por reunião.

As verificações desta seção funcionam como critérios de integridade. Elas não melhoram as métricas por si mesmas, mas impedem que a análise prossiga silenciosamente com IDs repetidos, campos ausentes ou relações inconsistentes.


In [5]:
seen_chunk_ids = set()
last_index_by_meeting = {}
observed_meetings = set()
validated_chunks = 0
for chunk in iter_ndjson(OUTPUT_PATH):
    assert chunk['chunk_id'] not in seen_chunk_ids
    assert 0 < chunk['num_tokens'] <= CONTENT_MAX_TOKENS
    expected_index = last_index_by_meeting.get(chunk['meeting_id'], -1) + 1
    assert chunk['chunk_index'] == expected_index
    seen_chunk_ids.add(chunk['chunk_id'])
    last_index_by_meeting[chunk['meeting_id']] = chunk['chunk_index']
    observed_meetings.add(chunk['meeting_id'])
    validated_chunks += 1

assert validated_chunks == summary['output_chunks']
assert len(observed_meetings) == summary['input_meetings']
{'validated_chunks': validated_chunks, 'validated_meetings': len(observed_meetings), 'unique_chunk_ids': len(seen_chunk_ids), 'max_tokens_observed': summary['chunk_tokens_max']}

{'validated_chunks': 29972,
 'validated_meetings': 1126,
 'unique_chunk_ids': 29972,
 'max_tokens_observed': 510}